In [ ]:
import pandas as pd
import pandas_bokeh
import seaborn as sns
import time

import panel as pn
pn.extension()

In [ ]:
import yfinance as yf
def get_data(ticker, n_days=1, interval=1):
    ticker = yf.Ticker(ticker)
    # see https://algotrading101.com/learn/yfinance-guide/ for more examples
    df = ticker.history(period=f'{n_days}D', interval=f'{interval}m')
    return df  

df = get_data('BTC-USD', n_days=7, interval=1)
df

In [ ]:
def resample_data(df, interval):
    df_resampled = df\
        .resample('60min')\
        .agg({'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'})
    return df_resampled

df_resampled = resample_data(df, '60min')

In [ ]:
from bokeh.plotting import figure, show

def plot_line(df, column, color, plot):
    plot.line(x=df.index, 
              y=df[column],
              line_width=2,
              legend_label=column,
              color=color,
              alpha=0.5)
    return plot

def plot_multiple_lines(df, columns, colors, plot):
    # plot = figure(x_axis_type="datetime")
    colors = ['blue', 'green', 'red', 'orange']
    for column, color in zip(columns, colors):
        plot = plot_line(df, column, color, plot)
    
    return plot




# show(plot_multiple_lines(df=df_resampled, 
#                 columns=['Open', 'Close'], 
#                 colors=['blue', 'green'], 
#                 plot=figure(x_axis_type="datetime"))) 

In [ ]:
def update_widgets(event):
    stock = stocks_panel.value
    df = get_data(stock, n_days=7, interval=1)

    columns = columns_panel.value
    
    log_pane.object = f'{time.ctime()}: Loaded data for {stock} with columns {columns}'
    
    plot = plot_multiple_lines(df=df, 
                columns=['Open', 'Close'], 
                colors=['blue', 'green'], 
                plot=figure(x_axis_type="datetime"))
    stock_evolution_panel.object = plot
    

In [ ]:
stocks_panel = pn.widgets.Select(
    options=['BTC-USD', 'ETH-USD', 'LTC-USD'],
    value=['BTC-USD'],
    # size=6,
    name='Cryptocurrencies'
)
pn.bind(update_widgets, stocks_panel, watch=True)

columns_panel = pn.widgets.MultiChoice(
    options=['Open', 'Close', 'High', 'Low'],
    value=['Open', 'Close'],
    name='Columns'
)
pn.bind(update_widgets, columns_panel, watch=True)



stock_evolution_panel = pn.pane.Bokeh(
    plot_multiple_lines(df=df_resampled, 
                columns=['Open', 'Close'], 
                colors=['blue', 'green'], 
                plot=figure(x_axis_type="datetime")),
    width=800,
    height=400
)

filter_panel = pn.layout.Column(
    stocks_panel,
    columns_panel
)

In [ ]:
pn.state.add_periodic_callback(update_widgets, period=100)

In [ ]:
log_pane = pn.pane.Markdown('??? info')
pn.layout.Row(
    filter_panel,
    # stock_evolution_panel,
    log_pane
).servable()